In [21]:
import cv2
import pytesseract
import re
import os

# 1. CONFIGURACIÓN DEL MOTOR (Ruta obligatoria en Windows)
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
RUTA_CARPETA = r'C:\Users\SUGLE-23\Documents\PRUEBA TESSERACT'

In [26]:
def extraer_coordenadas_gps(ruta_imagen):
    imagen = cv2.imread(ruta_imagen)
    if imagen is None: return "Error", "Error", ""

    # 1. Redimensionar para claridad
    imagen = cv2.resize(imagen, None, fx=1.5, fy=1.5, interpolation=cv2.INTER_LANCZOS4)
    
    # 2. Convertir a Gris y aplicar un desenfoque leve para eliminar "granulado" del pasto
    gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
    desenfoque = cv2.GaussianBlur(gris, (5, 5), 0)
    
    # 3. Umbral Adaptativo (Mejorado para detectar bordes de letras blancas)
    binaria = cv2.adaptiveThreshold(desenfoque, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                    cv2.THRESH_BINARY_INV, 15, 4)

    # 4. Configuración: PSM 11 es mejor para texto disperso en fotos de campo
    config_ocr = r'--psm 11 -c tessedit_char_whitelist=0123456789mEnN'
    texto = pytesseract.image_to_string(binaria, config=config_ocr)

    # 5. LÓGICA DE EXTRACCIÓN REFINADA
    # Buscamos específicamente el número que tiene 'mE' o 'mN' al lado
    este_search = re.search(r"(\d{6})\s*mE", texto, re.IGNORECASE)
    norte_search = re.search(r"(\d{7})\s*mN", texto, re.IGNORECASE)

    v_este = este_search.group(1) if este_search else "No detectado"
    v_norte = norte_search.group(1) if norte_search else "No detectado"

    # Plan B: Si no detectó mE/mN, buscamos números que cumplan el formato UTM
    if v_este == "No detectado" or v_norte == "No detectado":
        todos_los_numeros = re.findall(r"\d{6,7}", texto)
        for num in todos_los_numeros:
            if len(num) == 6 and v_este == "No detectado":
                v_este = num
            if len(num) == 7 and v_norte == "No detectado":
                v_norte = num

    return v_este, v_norte, texto

In [27]:
# 2. EJECUCIÓN DEL BUCLE
print(f"{'Archivo':<40} | {'Este':<12} | {'Norte':<12}")
print("-" * 75)

if os.path.exists(RUTA_CARPETA):
    for archivo in os.listdir(RUTA_CARPETA):
        if archivo.lower().endswith((".jpg", ".png")):
            ruta_completa = os.path.join(RUTA_CARPETA, archivo)
            este, norte, _ = extraer_coordenadas_gps(ruta_completa)
            print(f"{archivo:<40} | {este:<12} | {norte:<12}")
else:
    print(f"La ruta no existe: {RUTA_CARPETA}")

print("-" * 75)
print("Proceso finalizado.")

Archivo                                  | Este         | Norte       
---------------------------------------------------------------------------
20250828_083553.jpg                      | 501225       | No detectado
20250828_083646.jpg                      | No detectado | No detectado
20250828_083653.jpg                      | No detectado | No detectado
20250829_080614.jpg                      | 458257       | No detectado
---------------------------------------------------------------------------
Proceso finalizado.
